In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import os
import glob
from collections import Counter
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def mask_low_variation(signal, threshold=0.01):
    grad = np.abs(np.gradient(signal))
    mask = grad > threshold
    masked_signal = signal.copy()
    masked_signal[~mask] = np.nan
    return masked_signal

def process_filtered_signal(filename):
    df = pd.read_csv(os.path.join("data", filename))
    df['acc_total'] = np.sqrt(df['gFx']**2 + df['gFy']**2 + df['gFz']**2)

    # Extract time window from 10s to 15s
    start_time = df['time'].iloc[0] + 10
    end_time = start_time + 5
    df = df[(df['time'] >= start_time) & (df['time'] <= end_time)].reset_index(drop=True)

    # Estimate sampling frequency
    time_diff = np.diff(df['time'])
    fs = 1 / np.mean(time_diff) if len(time_diff) > 0 else 100

    # Apply filter and mask
    df['butter'] = butter_lowpass_filter(df['acc_total'], cutoff=5, fs=fs)
    df['butter_masked'] = mask_low_variation(df['butter'])

    return df['time'], df['butter_masked']

def load_all_ankle_data():
    all_features = []
    all_labels = []
    movement_types = ['run', 'walk', 'upstair', 'downstair']

    for movement in movement_types:
        files = glob.glob(f"data/{movement}*-ankle.csv")
        for file_path in files:
            filename = os.path.basename(file_path)
            try:
                time, masked_signal = process_filtered_signal(filename)
                clean_signal = masked_signal[~np.isnan(masked_signal)]
                if len(clean_signal) < 10:
                    continue

                feature_vector = [
                    np.nanmean(clean_signal),
                    np.nanstd(clean_signal),
                    np.nanmax(clean_signal) - np.nanmin(clean_signal),
                    np.nanmedian(clean_signal)
                ]
                all_features.append(feature_vector)

                if movement in ['upstair', 'downstair']:
                    all_labels.append('stairs')
                else:
                    all_labels.append(movement)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    return np.array(all_features), np.array(all_labels)

X, y = load_all_ankle_data()
print(f"\nTotal samples: {len(X)}, Feature dimensions: {X.shape[1]}")
print(f"Class distribution: {dict(Counter(y))}")

def evaluate_cv(model, name):
    scores = cross_val_score(model, X, y, cv=5)
    print(f"\n{name}:")
    print(f"Mean Accuracy: {scores.mean():.2f} (+/- {scores.std():.2f})")

evaluate_cv(RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest")
evaluate_cv(KNeighborsClassifier(n_neighbors=3), "KNN")
evaluate_cv(GaussianNB(), "Naive Bayes")


Total samples: 25, Feature dimensions: 4
Class distribution: {'run': 5, 'walk': 6, 'stairs': 14}

Random Forest:
Mean Accuracy: 0.96 (+/- 0.08)

KNN:
Mean Accuracy: 0.76 (+/- 0.15)

Naive Bayes:
Mean Accuracy: 0.84 (+/- 0.15)
